# Módulo 20: Fundamentos de LLMs
## Cuaderno Interactivo: Tokenización y Embeddings Vectoriales

**Diplomado Neurum 2026**  
*Autor: Docente Ever Torres, PhD*

---

### 🎯 Objetivos de este Cuaderno
1. **Comprender la Tokenización**: Experimentar cómo los modelos de lenguaje descomponen texto humano en subpalabras y Token IDs numéricos usando algoritmos como Byte-Pair Encoding (BPE).
2. **Explorar los Embeddings Vectoriales**: Transformar identificadores discretos en vectores continuos en un espacio geométrico de alta dimensión.
3. **Comprobar la Álgebra Semántica**: Medir la similitud de coseno entre conceptos y replicar operaciones míticas como $\text{Rey} - \text{Hombre} + \text{Mujer} \approx \text{Reina}$.
4. **Visualizar Espacios Semánticos**: Reducir dimensiones con PCA para graficar agrupaciones de conceptos (salud, tecnología, realeza, frutas) en 2D y 3D.

---


---
### 🛠️ 0. Instalación e Importación de Dependencias

Ejecuta la siguiente celda si estás corriendo este notebook en **Google Colab** o en un entorno virtual nuevo.


In [ ]:
# Si estás en Google Colab o entorno nuevo, descomenta esta línea:
# !pip install tiktoken transformers gensim scikit-learn matplotlib seaborn torch numpy

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn

print("✅ Dependencias principales cargadas correctamente.")


---
## Parte 1: Tokenización — De Texto a Números

Como vimos en la **Diapositiva 03 y 04**, las computadoras no entienden palabras ni letras sueltas.
- **Caracteres sueltos**: Generarían secuencias demasiado largas y la IA perdería el contexto.
- **Palabras completas**: Requerirían diccionarios infinitos en cada idioma para términos raros o médicos (ej. *electrocardiograma*).
- **Subpalabras (Tokens / BPE)**: Dividen palabras complejas en prefijos, raíces y sufijos reutilizables (las piezas LEGO del lenguaje).

### 1.1 Tokenización BPE con `tiktoken` (OpenAI / GPT-4o)


In [ ]:
import tiktoken

# Cargamos el codificador BPE utilizado por modelos modernos como GPT-4 / GPT-4o
enc = tiktoken.get_encoding("cl100k_base")

texto_ejemplo = "El neurocirujano realizó una resonancia magnética en el hospital."

# Convertimos texto a lista de Token IDs
tokens = enc.encode(texto_ejemplo)

print(f"Texto original: '{texto_ejemplo}'")
print(f"Número de caracteres: {len(texto_ejemplo)}")
print(f"Número de tokens: {len(tokens)}")
print(f"Lista de Token IDs: {tokens}
")

print("--- Desglose Token por Token ---")
for token_id in tokens:
    # Decodificamos cada token individualmente
    subpalabra = enc.decode([token_id])
    print(f"ID: {token_id:<6} -> Subpalabra: '{subpalabra}'")


### 1.2 Comparación de Eficiencia: Español vs. Inglés

Debido a que muchos tokenizadores fueron entrenados mayoritariamente con texto en inglés, las frases en español a menudo se dividen en más tokens. Observemos este fenómeno a continuación:


In [ ]:
texto_es = "El paciente presentó síntomas de hipertensión arterial acelerada."
texto_en = "The patient presented symptoms of accelerated arterial hypertension."

tokens_es = enc.encode(texto_es)
tokens_en = enc.encode(texto_en)

print(f"Español: {len(texto_es)} caracteres -> {len(tokens_es)} tokens")
print(f"Tokens ES: {[enc.decode([t]) for t in tokens_es]}
")

print(f"Inglés:  {len(texto_en)} caracteres -> {len(tokens_en)} tokens")
print(f"Tokens EN: {[enc.decode([t]) for t in tokens_en]}")


### 1.3 Tokenización con HuggingFace Transformers (Llama-3 / BERT)


In [ ]:
from transformers import AutoTokenizer

# Cargamos un tokenizador de HuggingFace (BERT en español)
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

texto_medico = "Diagnóstico: Hipertensión arterial secundaria."

tokens_bert = tokenizer.tokenize(texto_medico)
ids_bert = tokenizer.convert_tokens_to_ids(tokens_bert)

print("Texto:", texto_medico)
print("Tokens subpalabra (BERT ES):", tokens_bert)
print("Token IDs:", ids_bert)


---
## Parte 2: Embeddings Vectoriales — El Mapa del Significado

Como vimos en la **Diapositiva 06 y 07**, un número por sí solo (`ID: 4821`) no transmite significado. 
Un **Embedding** convierte cada Token ID en un vector continuo $d$-dimensional en un **GPS semántico**, donde conceptos afines están cerca geométricamente.

### 2.1 Creación de una Capa de Embedding en PyTorch (`nn.Embedding`)


In [ ]:
# Supongamos un vocabulario pequeño de 10,000 tokens y dimensión de embedding d = 16
vocab_size = 10000
embedding_dim = 16

# Capa de embedding (tabla de búsqueda entrenable)
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Tomamos una secuencia de Token IDs
input_ids = torch.tensor([3491, 812, 9401]) # ej: ["hospital", "neuro", "logía"]

# Obtenemos los vectores de embedding
vectors = embedding_layer(input_ids)

print("Forma de la matriz de tokens de entrada:", input_ids.shape)
print("Forma del tensor de embeddings resultante:", vectors.shape)
print("
Vector continuo para el primer token (dimensiones 16d):")
print(vectors[0].detach().numpy().round(3))


### 2.2 Carga de Embeddings Semánticos Preentrenados (`gensim`)


In [ ]:
import gensim.downloader as api

print("⏳ Cargando modelo preentrenado de embeddings (glove-wiki-gigaword-100)...")
# Carga modelo compacto de 100 dimensiones
model = api.load("glove-wiki-gigaword-100")
print("✅ Modelo cargado con éxito.")

# Dimensión del vector para una palabra
vec_doctor = model["doctor"]
print(f"
Dimensión del vector 'doctor': {len(vec_doctor)}")
print("Primeros 10 valores del vector:", vec_doctor[:10].round(3))


### 2.3 Similitud de Coseno entre Palabras

Medimos el ángulo entre vectores para saber qué tan cerca están conceptualmente.


In [ ]:
def calcular_similitud(palabra1, palabra2):
    if palabra1 in model and palabra2 in model:
        vec1 = model[palabra1].reshape(1, -1)
        vec2 = model[palabra2].reshape(1, -1)
        sim = cosine_similarity(vec1, vec2)[0][0]
        print(f"Similitud entre '{palabra1}' y '{palabra2}': {sim * 100:.2f}%")
        return sim
    else:
        print(f"Una de las palabras ('{palabra1}' o '{palabra2}') no está en el vocabulario.")

# Pruebas de similitud
calcular_similitud("doctor", "hospital")
calcular_similitud("doctor", "physician")
calcular_similitud("doctor", "apple")
calcular_similitud("king", "queen")


---
## Parte 3: Álgebra Semántica y Visualización en 2D/3D

### 3.1 Álgebra Vectorial: $\text{Rey} - \text{Hombre} + \text{Mujer} = \text{Reina}$


In [ ]:
# Operación de álgebra vectorial
v_king = model["king"]
v_man = model["man"]
v_woman = model["woman"]
v_queen = model["queen"]

# Resultado matemático: Rey - Hombre + Mujer
v_result = v_king - v_man + v_woman

# Encontrando las palabras más cercanas al vector resultante
palabras_cercanas = model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

print("Resultados de la ecuación semántica (King - Man + Woman):")
for palabra, score in palabras_cercanas:
    print(f" -> {palabra:<12} (Similitud: {score*100:.2f}%)")


### 3.2 Visualización en 3D del Espacio Vectorial con PCA

Reducimos los vectores de 100 dimensiones a 3 dimensiones con PCA para proyectar los puntos en un gráfico 3D interactivo.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

palabras = ["king", "man", "queen", "woman", "doctor", "hospital", "apple", "fruit"]
vectores = [model[p] for p in palabras]

# Agregamos también el vector del resultado algebraico
vectores.append(v_result)
palabras.append("RESULTADO (king-man+woman)")

# PCA a 3 componentes principales
pca = PCA(n_components=3)
coords_3d = pca.fit_transform(np.array(vectores))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Graficar puntos
for i, p in enumerate(palabras):
    x, y, z = coords_3d[i]
    color = 'red' if 'RESULTADO' in p else ('blue' if p in ['king', 'queen', 'man', 'woman'] else 'green')
    ax.scatter(x, y, z, color=color, s=80)
    ax.text(x + 0.02, y + 0.02, z + 0.02, p, fontsize=10, fontweight='bold')

ax.set_title("Proyección 3D de Embeddings Vectoriales (PCA)", fontsize=14, pad=15)
ax.set_xlabel("Componente PCA 1")
ax.set_ylabel("Componente PCA 2")
ax.set_zlabel("Componente PCA 3")
plt.tight_layout()
plt.show()


### 3.3 Visualización en 2D por Clusters Temáticos (Salud vs. Frutas vs. Tecnología)


In [ ]:
# Agrupaciones de palabras por dominio
cluster_salud = ["doctor", "hospital", "nurse", "medicine", "patient"]
cluster_frutas = ["apple", "banana", "orange", "fruit", "grape"]
cluster_tech = ["computer", "software", "internet", "algorithm", "python"]

todas_palabras = cluster_salud + cluster_frutas + cluster_tech
vectores_cluster = [model[w] for w in todas_palabras]

pca_2d = PCA(n_components=2)
coords_2d = pca_2d.fit_transform(np.array(vectores_cluster))

plt.figure(figsize=(11, 7))

for i, word in enumerate(todas_palabras):
    x, y = coords_2d[i]
    if word in cluster_salud:
        color = '#2563EB'
        marker = 'o'
    elif word in cluster_frutas:
        color = '#10B981'
        marker = 's'
    else:
        color = '#7C3AED'
        marker = '^'
        
    plt.scatter(x, y, color=color, marker=marker, s=120, alpha=0.85)
    plt.text(x + 0.05, y + 0.05, word, fontsize=11, fontweight='semibold')

plt.title("GPS del Significado: Clusters Semánticos en 2D (PCA)", fontsize=14, fontweight='bold')
plt.xlabel("Dimensión Semántica 1")
plt.ylabel("Dimensión Semántica 2")
plt.grid(True, linestyle='--', alpha=0.5)

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Salud / Medicina', markerfacecolor='#2563EB', markersize=10),
    Line2D([0], [0], marker='s', color='w', label='Frutas / Alimentos', markerfacecolor='#10B981', markersize=10),
    Line2D([0], [0], marker='^', color='w', label='Tecnología / Cómputo', markerfacecolor='#7C3AED', markersize=10)
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()


---
## 💡 Resumen y Conclusiones del Cuaderno

1. **Tokenización**: Los LLMs dividen las oraciones en subpalabras (tokens) para manejar eficientemente palabras frecuentes, técnicas y desconocidas sin saturar el tamaño del vocabulario.
2. **Embeddings**: Asignan a cada Token ID una representación vectorial continua de alta dimensión donde la distancia geométrica representa cercanía semántica.
3. **Álgebra Semántica**: Las propiedades del espacio vectorial permiten hacer operaciones numéricas con significado (ej. restar género o sumar plurales/conceptos).
4. **Reducción de Dimensiones (PCA)**: Permite proyectar mapas multidimensionales a planos 2D o 3D verificando visualmente los agrupamientos conceptuales.

---
